::: {.sb-downloads}
[Download this notebook](../downloads/solutions-electricity.ipynb){download="solutions-electricity.ipynb"}  
Data: [pjme_hourly.csv](../downloads/data/pjme_hourly.csv){download="pjme_hourly.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

::: {.sb-lesson-links}
[← Back to the exercises](../exercises/electricity.ipynb)
:::

::: {.callout-tip}
Try the exercises on your own first: you learn much more by searching than by reading a solution.
:::

[PJM](https://www.pjm.com/) runs the electricity grid of a large part of the eastern United States. To keep the grid balanced, the operator must know in advance how much electricity people will use, hour by hour: too little production means blackouts, too much is wasted money.

`../data/pjme_hourly.csv` contains the hourly consumption of the *PJM East* region, in megawatts (MW), from 2002 to 2018. In this exercise, you'll forecast one full year, hour by hour, with Prophet.

In [ ]:
import logging
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet
from prophet.plot import plot_components_plotly
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

logging.getLogger("cmdstanpy").disabled = True  # hide the messages of Prophet's optimizer

import plotly.io as pio
import plotly.graph_objects as go
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

## Exploration {#ex-exploration}

1. Load the data with `index_col=0, parse_dates=[0]`, sort it by date and remove the duplicated timestamps. Keep the years 2011 to 2015.

In [ ]:
energy = pd.read_csv("../data/pjme_hourly.csv", index_col=0, parse_dates=[0]).sort_index()
energy = energy[~energy.index.duplicated()]
energy = energy.loc["2011":"2015"]
print(energy.shape)
energy.head()

(43816, 1)


,PJME_MW
Datetime,
2011-01-01 00:00:00,29871.0
2011-01-01 01:00:00,28638.0
2011-01-01 02:00:00,27630.0
2011-01-01 03:00:00,26838.0
2011-01-01 04:00:00,26443.0


2. Plot the consumption over time. With more than 40,000 hourly values, the plot is unreadable: plot the **daily average** instead (`resample("D").mean()`).

In [ ]:
daily = energy.resample("D").mean()
fig = px.line(daily, y="PJME_MW", title="Daily average consumption (MW)")
fig.show()

3. Plot the average consumption by hour of the day, by day of the week and by month. What patterns do you see?

In [ ]:
by = {
    "hour of the day": energy.index.hour,
    "day of the week (0 = Monday)": energy.index.day_of_week,
    "month": energy.index.month,
}
for name, key in by.items():
    profile = energy.groupby(key)["PJME_MW"].mean()
    fig = px.bar(x=profile.index, y=profile.values, labels={"x": name, "y": "average consumption (MW)"},
                 title=f"Average consumption by {name}")
    fig.show()

**Three seasonalities, as expected for electricity:**

* **daily**: low at night (around 4 a.m.), high in the evening;
* **weekly**: lower on weekends, when offices and factories are closed;
* **yearly, with two peaks**: summer (air conditioning) and winter (heating), and lows in spring and autumn.

## A first Prophet model {#ex-a-first-prophet-model}

4. Train on 2011-2014 and keep 2015 as the test set. Prophet needs two columns: `ds` (the timestamps) and `y` (the values).

In [ ]:
data = energy.reset_index().rename(columns={"Datetime": "ds", "PJME_MW": "y"})
train = data[data["ds"] < "2015-01-01"]
test = data[data["ds"] >= "2015-01-01"]
print("train:", train.shape, " test:", test.shape)

train: (35057, 2)  test: (8759, 2)


5. Train a Prophet model and forecast the test period. Then plot the components of the model: did it find the patterns of question 3?

In [ ]:
model = Prophet()
model.fit(train)
forecast = model.predict(test[["ds"]])

fig = plot_components_plotly(model, forecast)
fig.show()

**Prophet found the three seasonalities: the daily cycle, the weekly cycle (lower on weekends) and the yearly cycle with its two peaks.**

6. Plot the forecast and the real consumption for the first week of 2015, then compute the MAE and the MAPE on the whole test year.

In [ ]:
def plot_period(start, end, forecasts, title):
    """Real consumption and one or several forecasts between two dates."""
    real = test[(test["ds"] >= start) & (test["ds"] < end)]
    fig = go.Figure(go.Scatter(x=real["ds"], y=real["y"], mode="lines", name="real", line=dict(color="#111111")))
    for name, fc in forecasts.items():
        part = fc[(fc["ds"] >= start) & (fc["ds"] < end)]
        fig.add_trace(go.Scatter(x=part["ds"], y=part["yhat"], mode="lines", name=name))
    fig.update_layout(title=title, yaxis_title="MW", template="plotly_white")
    return fig


plot_period("2015-01-01", "2015-01-08", {"Prophet": forecast}, "First week of 2015").show()

print(f"MAE:  {mean_absolute_error(test['y'], forecast['yhat']):,.0f} MW")
print(f"MAPE: {mean_absolute_percentage_error(test['y'], forecast['yhat']):.1%}")

MAE:  3,458 MW
MAPE: 10.3%


**The forecast follows the daily cycle well, a whole year in advance. The error is about 10% on average: Prophet doesn't know the weather, and the weather drives a large part of the consumption (a cold week in winter, a heat wave in summer).**

## Holidays {#ex-holidays}

7. On public holidays, offices and factories are closed: the consumption looks like a Sunday. Train a second model with the US holidays (`add_country_holidays(country_name="US")`), and compare the two models on the week of July 4th, 2015.

In [ ]:
model_holidays = Prophet()
model_holidays.add_country_holidays(country_name="US")
model_holidays.fit(train)
forecast_holidays = model_holidays.predict(test[["ds"]])

plot_period("2015-07-01", "2015-07-08", {"without holidays": forecast, "with holidays": forecast_holidays},
            "Week of July 4th, 2015").show()

8. Compute the MAE of both models on all the test year, on the public holidays only (use `USFederalHolidayCalendar`), and on July 4th only. Conclude.

In [ ]:
holidays_2015 = USFederalHolidayCalendar().holidays(start="2015-01-01", end="2015-12-31")
on_holiday = test["ds"].dt.normalize().isin(holidays_2015).to_numpy()
on_july_4 = (test["ds"].dt.strftime("%m-%d") == "07-04").to_numpy()

rows = []
for name, fc in {"without holidays": forecast, "with holidays": forecast_holidays}.items():
    rows.append({"model": name,
                 "MAE, whole year": mean_absolute_error(test["y"], fc["yhat"]),
                 "MAE, public holidays": mean_absolute_error(test["y"][on_holiday], fc["yhat"][on_holiday]),
                 "MAE, July 4th": mean_absolute_error(test["y"][on_july_4], fc["yhat"][on_july_4])})
pd.DataFrame(rows).set_index("model").round(0)

,"MAE, whole year","MAE, public holidays","MAE, July 4th"
model,,,
without holidays,3458.0,3974.0,3816.0
with holidays,3623.0,3338.0,1228.0


**With the holidays, the error on public holidays decreases, and it is divided by about three on July 4th. But the error on the whole year slightly increases! The holiday effects are estimated from only four years, so they are not perfect, and they also change the rest of the model a little.**

**Lesson: always check a model improvement where it matters for the business. If the grid operator mostly fears the errors on special days, the holiday model is better; for the average day, the simpler one is as good. To go much further, add what really drives the consumption: the temperature (as a regressor, with the weather forecast for the future dates).**